# Transformasi Sumbu Y menggunakan Translasi dan Pencerminan

**Blok Utama** (Biru): posisi asli titik-titik koordinat GeoGebra Anda.  
**Blok Cermin** (Merah): hasil setelah mengalami translasi (pergeseran vertikal ke atas) dan refleksi (pencerminan atas-bawah terhadap sumbu X).

### Persamaan Matematika (Koordinat Homogen):
1. **Translasi ke atas sejauh $t$ satuan ($t_x = 0, t_y = t$):**
   $$T = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & t \\ 0 & 0 & 1 \end{bmatrix}$$
2. **Refleksi terhadap Sumbu X / Atas-Bawah ($s_y = s$, dengan $s$ turun dari $1 \to -1$):**
   $$M = \begin{bmatrix} 1 & 0 & 0 \\ 0 & s & 0 \\ 0 & 0 & 1 \end{bmatrix}$$
3. **Transformasi Gabungan ($A = M \cdot T$):**
   $$A = \begin{bmatrix} 1 & 0 & 0 \\ 0 & s & 0 \\ 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & t \\ 0 & 0 & 1 \end{bmatrix} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & s & s \cdot t \\ 0 & 0 & 1 \end{bmatrix}$$\n

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from IPython.display import HTML, display\n

In [ ]:
# ── Titik-titik dari GeoGebra Anda ──────────────────────────────
TITIK = {
    'A': (2, 3),  'B': (2, 4),  'C': (3, 4),  'D': (3, 3),
    'E': (2,-3),  'F': (3,-3),  'G': (2,-4),  'H': (3,-4),
    'I': (2, 2),  'J': (3, 2),  'K': (2, 1),  'L': (3, 1),
    'M': (2,-1),  'N': (3,-1),  'O': (2,-2),  'P': (3,-2),
}

FRAMES  = 160
PAUSE_F = 30

# ── Setup figure ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 8))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_aspect('equal')
ax.axhline(0, color='black', linewidth=1.2, zorder=2)
ax.axvline(0, color='black', linewidth=1.2, zorder=2)
ax.grid(True, color='#cccccc', linewidth=0.5)
ax.tick_params(labelsize=9)
ax.set_title('Animasi Translasi ke Atas dan Pencerminan terhadap Sumbu X (Atas-Bawah)\n'
             '(Klik grafik untuk Pause/Play)',
             fontsize=11, pad=10)

# ── Warna per pasangan blok Anda ─────────────────────────────────
BLOK = [
    {'asli': ['A','B','C','D'], 'warna': '#1a6fb5'},
    {'asli': ['I','J','L','K'], 'warna': '#7c3aed'},
    {'asli': ['M','N','P','O'], 'warna': '#d97706'},
]

# artists per blok
blok_artists = []
for blok in BLOK:
    warna = blok['warna']
    pts   = [TITIK[k] for k in blok['asli']]
    xs    = [p[0] for p in pts]
    ys    = [p[1] for p in pts]

    # Kotak asli
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    r_asli = patches.Rectangle(
        (xmin, ymin), xmax-xmin, ymax-ymin,
        lw=2, edgecolor=warna, facecolor='none', zorder=5
    )
    ax.add_patch(r_asli)
    d_asli, = ax.plot([], [], 'o', color=warna, ms=7, zorder=6)
    t_asli  = [ax.text(0,0, lbl, color=warna, fontsize=8,
                       fontweight='bold', zorder=7)
               for lbl in blok['asli']]

    # Kotak cermin
    r_cermin = patches.Rectangle(
        (xmin, ymin), xmax-xmin, ymax-ymin,
        lw=2, edgecolor='#e74c3c', facecolor='none', zorder=5
    )
    ax.add_patch(r_cermin)
    d_cermin, = ax.plot([], [], 'o', color='#e74c3c', ms=7, zorder=6)
    lbl_cermin = [k+"'" for k in blok['asli']]
    t_cermin   = [ax.text(0,0, lbl, color='#e74c3c', fontsize=8,
                          fontweight='bold', zorder=7)
                  for lbl in lbl_cermin]

    blok_artists.append({
        'r_asli': r_asli, 'd_asli': d_asli, 't_asli': t_asli,
        'r_cermin': r_cermin, 'd_cermin': d_cermin, 't_cermin': t_cermin,
        'xmin': xmin, 'xmax': xmax, 'ymin': ymin, 'ymax': ymax
    })

# ── Info teks matriks ────────────────────────────────────────────
mat_txt = ax.text(-5.8, 4.0, '', fontsize=9, color='#222',
                  fontfamily='monospace',
                  bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f4ff',
                            edgecolor='#aaaaaa', alpha=0.9))

# ── Legend ───────────────────────────────────────────────────────
legend_elements = [
    Line2D([0],[0], color='#1a6fb5', lw=2, label='Blok Asli'),
    Line2D([0],[0], color='#e74c3c', lw=2, label="Blok Cermin (P')"),
]
ax.legend(handles=legend_elements, loc='upper right',
          fontsize=9, framealpha=0.9)

# ── State pause ──────────────────────────────────────────────────
state = {'paused': False}
def on_click(event):
    if event.inaxes == ax:
        state['paused'] = not state['paused']
fig.canvas.mpl_connect('button_press_event', on_click)

# ── Easing scale dan translasi ──────────────────────────────────
def get_scale_and_trans(frame):
    if frame < PAUSE_F:
        return 0.0, 1.0
    if frame >= FRAMES - PAUSE_F:
        return 1.0, -1.0
    
    mid = 80
    if frame < mid:
        t_val = (frame - PAUSE_F) / (mid - PAUSE_F - 1)
        t_val = 3*t_val**2 - 2*t_val**3
        return t_val, 1.0
    else:
        s_val = (frame - mid) / (FRAMES - PAUSE_F - mid)
        s_val = 3*s_val**2 - 2*s_val**3
        s = 1.0 - 2.0 * s_val
        return 1.0, s

cur_frame = [0]

def animate(frame):
    if state['paused']:
        frame = cur_frame[0]
    else:
        cur_frame[0] = frame

    t, s = get_scale_and_trans(frame)
    all_artists = [mat_txt]

    for ba in blok_artists:
        xmin = ba['xmin']; xmax = ba['xmax']
        ymin = ba['ymin']; ymax = ba['ymax']

        # Blok asli: y dikompres/direfleksikan ke sumbu X
        y0a = ymin * s
        y1a = ymax * s
        ya_min = min(y0a, y1a)
        ha = abs(y0a - y1a)
        ba['r_asli'].set_y(ya_min)
        ba['r_asli'].set_height(max(ha, 1e-6))

        # sudut: BL, BR, TR, TL
        cx = [xmin, xmax, xmax, xmin]
        cy_a = [y0a, y0a, y1a, y1a]
        ba['d_asli'].set_data(cx, cy_a)
        off = [(-0.15,-0.2),(0.05,-0.2),(0.05,0.05),(-0.15,0.05)]
        for tx, px, py, (dx, dy) in zip(ba['t_asli'], cx, cy_a, off):
            tx.set_position((px+dx, py+dy))
        all_artists += [ba['d_asli'], ba['r_asli']] + ba['t_asli']

        # Blok cermin: translasi (+t) lalu refleksi (*s)
        y0c = (ymin + t) * s
        y1c = (ymax + t) * s
        yc_min = min(y0c, y1c)
        hc = abs(y0c - y1c)
        ba['r_cermin'].set_y(yc_min)
        ba['r_cermin'].set_height(max(hc, 1e-6))
        
        cy_c = [y0c, y0c, y1c, y1c]
        off2 = [(-0.15,-0.25),(0.05,-0.25),(0.05,0.05),(-0.15,0.05)]
        for tx, px, py, (dx, dy) in zip(ba['t_cermin'], cx, cy_c, off2):
            tx.set_position((px+dx, py+dy))
        ba['d_cermin'].set_data(cx, cy_c)
        all_artists += [ba['d_cermin'], ba['r_cermin']] + ba['t_cermin']

    mat_txt.set_text(
        f'T_trans = [1.0  0.0  0.00]\n'
        f'          [0.0  1.0  {t:.2f}]\n'
        f'          [0.0  0.0  1.00]\n\n'
        f'M_refly = [1.0  0.0  0.00]\n'
        f'          [0.0  {s:.2f}  0.00]\n'
        f'          [0.0  0.0  1.00]\n\n'
        f'A_gab   = [1.0  0.0  0.00]\n'
        f'          [0.0  {s:.2f}  {s*t:.2f}]\n'
        f'          [0.0  0.0  1.00]'
    )
    return all_artists

ani = animation.FuncAnimation(
    fig, animate, frames=FRAMES,
    interval=45, blit=True, repeat=True
)

plt.tight_layout()
display(HTML(ani.to_jshtml()))
plt.close()\n

## Tabel Koordinat Transformasi Sumbu Y

| Titik Asli | Hasil Translasi $T(y+1)$ | Hasil Akhir Refleksi $P''(-y-1)$ |
|:---:|:---:|:---:|
| $A(2,3)$ | $A'(2,4)$ | $A''(2,-4)$ |
| $B(2,4)$ | $B'(2,5)$ | $B''(2,-5)$ |
| $C(3,4)$ | $C'(3,5)$ | $C''(3,-5)$ |
| $D(3,3)$ | $D'(3,4)$ | $D''(3,-4)$ |
| $I(2,2)$ | $I'(2,3)$ | $I''(2,-3)$ |
| $J(3,2)$ | $J'(3,3)$ | $J''(3,-3)$ |
| $K(2,1)$ | $K'(2,2)$ | $K''(2,-2)$ |
| $L(3,1)$ | $L'(3,2)$ | $L''(3,-2)$ |
\n